# 📙 [참고] A/B 테스트 — 데이터 기반 의사결정

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

**이 노트북은 참고 자료입니다.** 9일차 본 교안은 **가설검정(교안 01)** 과 **회귀분석(교안 02)** 이고, 여기 담긴 **A/B 테스트**는 그 도구들을 **비즈니스 의사결정**에 쓰는 응용편이라 따로 묶었습니다. 본 교안·과제를 마친 뒤 이어서 보면 됩니다.

## 이 노트북의 구성
| 파트 | 내용 |
|---|---|
| **교안 (1~4절)** | A/B 테스트 개념 · 비율 z-검정 · 효과크기 Cohen's h · 신뢰구간 · 표본크기 산정 · 통계적 유의 ≠ 실질적 유의 |
| **실습 문제 (2문)** | 실제 이커머스 A/B 로그 294,478행으로 **불러오기 → 정제 → 전환율 비교 → 비율 검정 → 신뢰구간 → 의사결정** 을 처음부터 끝까지 |

## 풀이 방법
1. 교안 파트는 **🖐️ 함께 따라하기** 셀을 직접 채우며 읽습니다.
2. 실습 문제는 **1단계에서 데이터를 불러와** 같은 `df` 로 끝까지 이어 분석합니다(정제 결과가 뒤 단계로 이어집니다).
3. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채웁니다. 정량 단계는 아래 **자가채점 셀**로 확인하고, **그래프 단계는 자가채점 없이** 위 **완성 그래프(정답)** 와 같은 모양으로 그립니다.
4. **의사결정·해석 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

화이팅!

> 🔧 **이 단원의 도구**: 두 그룹의 **전환율(비율)** 을 비교하는 검정은 **2-비율 z-검정**입니다. 이건 이번 단원 주력인 **Pingouin 이 지원하지 않아**, 표준 도구인 **`statsmodels`**(`proportions_ztest`·`proportion_confint`)를 씁니다(효과크기는 **Cohen's h** `proportion_effectsize`). 앞서 배운 t-검정·ANOVA·카이제곱은 Pingouin, **회귀·비율검정은 statsmodels** 로 가는 경계를 그대로 따르는 것입니다.

In [ ]:
# [제공 코드] 통계 검정에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest, proportion_effectsize, proportion_confint,
)
from statsmodels.stats.power import NormalIndPower

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={"axes.unicode_minus": False})

---
# 1. A/B 테스트 — 감이 아니라 데이터로 정한다

## 왜 필요할까요?
"새 랜딩 페이지가 기존보다 더 잘 팔릴까?" 같은 물음에 감이 아니라 **데이터로** 답하는 것이 **A/B 테스트**입니다. 사용자를 무작위로 두 그룹으로 나눠(A=기존, B=신규) 각각에 다른 버전을 보여 주고, **전환율(구매·클릭 비율)** 을 비교합니다. 무작위 배정 덕분에 두 그룹의 다른 조건이 평균적으로 같아져 **차이를 '신규 페이지의 효과'로 인과적으로 해석**할 수 있습니다(회귀 교안 1절에서 '상관은 인과가 아니다'라며 미뤄 둔 이야기가 여기서 완성됩니다).

<img src="images/AB테스트_프로세스.png" width="720"/>

> 🔧 **왜 statsmodels?** 두 비율(전환율)의 z-검정은 **Pingouin 에 없습니다.** 그래서 A/B 비율 검정은 회귀와 마찬가지로 `statsmodels` 도구들로 합니다.

비율(전환율) 비교에 필요한 statsmodels 도구 네 가지를 씁니다. **무엇을 돌려주고 그 값을 어떻게 읽는지**까지 함께 봅니다.

| | 도구 | 인자 | **반환값과 읽는 법** |
|---|---|---|---|
| ⭐ | `proportions_ztest` | `count=[전환수1, 전환수2]`, `nobs=[방문수1, 방문수2]` | **`(z, p)` 두 값**. **`p` 가 결론**(`< 0.05` 면 전환율 차이가 유의). `z` 는 **부호만**(앞에 넣은 그룹이 높으면 +) 보면 됩니다 |
| ⭐ | `proportion_effectsize` | `(비율1, 비율2)` | **Cohen's h 하나** — 차이의 **크기**: **0.2 작음 / 0.5 중간 / 0.8 큼**(p 와 별개) |
| ○ | `proportion_confint` | `(전환수, 방문수, alpha=0.05)` | **`(하한, 상한)` 튜플** — 그 그룹 **전환율의 95% 신뢰구간**(불확실성의 폭) |
| ○ | `NormalIndPower().solve_power` | `effect_size=h`, `alpha`, `power` | **그룹당 필요한 표본 수**(소수 → 올림). 결론이 아니라 **실험 설계용** |

> **⭐ 결론에 쓰는 것**: **`p`**(유의한가) · **Cohen's h**(얼마나 큰가) · **실제 전환율**(어느 쪽이 높은가) — 늘 이 세 가지입니다.

---
# 2. 시나리오 — 랜딩 페이지 A/B 테스트

기존 페이지(A=대조군)와 신규 페이지(B=실험군)를 각각 **1,000명**에게 보여 준 결과입니다.

| 그룹 | 방문자 | 전환(구매) | 전환율 |
|---|---|---|---|
| A 대조군(기존) | 1,000 | 100 | 10.0% |
| B 실험군(신규) | 1,000 | 130 | 13.0% |

겉보기엔 13% > 10% 로 신규가 나아 보입니다. 이 **3%p 차이가 우연인지**, 그리고 **얼마나 의미 있는 차이인지**를 통계로 확인합니다.

In [ ]:
# A/B 데이터 (방문자 수와 전환 수)
n_control, x_control = 1000, 100      # 대조군: 전환율 10.0%
n_treat, x_treat = 1000, 130          # 실험군: 전환율 13.0%
rate_control = x_control / n_control
rate_treat = x_treat / n_treat
print('대조군 전환율 = %.3f, 실험군 전환율 = %.3f' % (rate_control, rate_treat))
print('절대 차이 = %.3f (%.1f%%p), 상대 향상 = %.1f%%' %
      (rate_treat - rate_control, (rate_treat - rate_control) * 100,
       (rate_treat - rate_control) / rate_control * 100))

# ① 비율의 z-검정: 두 전환율 차이가 유의한가 (실험군을 앞에 두면 z>0 이 '실험군이 높다')
z_stat, p_val = proportions_ztest(count=[x_treat, x_control], nobs=[n_treat, n_control])
print('\n[비율 z-검정] z = %.4f, p-value = %.4f' % (z_stat, p_val))
print('p < 0.05 → 전환율 차이가 통계적으로 유의' if p_val < 0.05 else 'p >= 0.05 → 유의하지 않음')

# ② 효과크기 Cohen's h
cohens_h = proportion_effectsize(rate_treat, rate_control)
print('\n[효과크기] Cohen h = %.4f  (0.2 미만=작음, 0.5=중간, 0.8=큼)' % cohens_h)

# ③ 각 그룹 전환율의 95% 신뢰구간
ci_control = proportion_confint(x_control, n_control, alpha=0.05, method='normal')
ci_treat = proportion_confint(x_treat, n_treat, alpha=0.05, method='normal')
print('\n[신뢰구간] 대조군 95%% CI = [%.4f, %.4f]' % ci_control)
print('[신뢰구간] 실험군 95%% CI = [%.4f, %.4f]' % ci_treat)

### 📊 A/B 결과를 결론으로 바꾸기 — ⭐ 세 개면 끝

위 출력의 숫자를 그대로 읽어 봅니다. **유의성(p) · 크기(h) · 방향(실제 전환율)** 이 전부입니다.

1. **유의한가?** → `z = 2.10`, `p = 0.0355` < 0.05 → **유의**합니다(두 페이지에 **차이가 없다면** 이만큼 벌어진 결과가 나올 확률이 3.6% 뿐 — 우연으로 보기엔 드뭅니다). z 가 **양수**인 것은 앞에 넣은 **실험군의 전환율이 더 높다**는 뜻입니다.
2. **얼마나 큰가?** → **Cohen's h = 0.094** → 기준(0.2/0.5/0.8)으로 보면 **작은 효과**입니다. "유의하다"와 "크다"는 다른 말임을 여기서도 확인합니다.
3. **어느 방향인가?** → 전환율 **10.0% → 13.0%**(절대 +3%p, 상대 +30%). 신뢰구간은 대조군 [8.1%, 11.9%], 실험군 [10.9%, 15.1%] 입니다.

> **주의**: 두 신뢰구간이 조금 **겹치지만**(11.9% vs 10.9%) 검정은 **유의**합니다. 구간이 겹친다고 "차이 없다"고 단정하면 안 됩니다 — 유의성 판정은 **차이에 대한 검정(p)** 으로 합니다.

> **결론 문장**: "신규 페이지의 전환율이 통계적으로 유의하게 높지만(p=0.036), 효과크기는 작습니다(h=0.09). 상대 향상 30%가 사업적으로 크다면 도입을 검토하되, 표본을 더 모아 재현되는지 확인하는 편이 안전합니다."

---
# 3. 표본크기 산정 — 실험을 시작하기 *전에* 정한다

A/B 테스트에서 가장 흔한 실수는 **결과를 계속 훔쳐보다가(peeking) 유의해지는 순간 멈추는 것**입니다. 이러면 우연한 차이를 효과로 오해하기 쉽습니다. 올바른 방법은 **실험 전에 필요한 표본크기를 정하고, 그만큼 모을 때까지 결과를 보지 않는 것**입니다.

표본크기는 네 가지로 정해집니다.
- **유의수준 α**(보통 0.05): 1종 오류(효과가 없는데 있다고 판정) 허용 확률
- **검정력 power**(보통 0.80): 진짜 효과가 있을 때 그것을 **잡아낼** 확률
- **기준 전환율**(현재 값, 예: 10%)
- **MDE(최소 검출 효과, Minimum Detectable Effect)**: 사업적으로 의미 있다고 볼 **최소한의 차이**(예: +1%p)

`NormalIndPower().solve_power` 로 "기준 10% 에서 +1%p(→11%) 를 검출하려면 그룹당 몇 명이 필요한가"를 구합니다.

In [ ]:
# 기준 전환율 10% 에서 +1%p(11%) 차이를 검출하려면 그룹당 몇 명?
baseline = 0.10
mde = 0.01                              # 최소 검출 효과: 1%p
effect_h = proportion_effectsize(baseline + mde, baseline)   # 목표 차이의 Cohen's h
n_needed = NormalIndPower().solve_power(effect_size=effect_h, alpha=0.05,
                                        power=0.80, alternative='two-sided')
print('목표 차이의 효과크기 h = %.4f' % effect_h)
print('그룹당 필요한 표본크기 = %d 명 (올림)' % int(np.ceil(n_needed)))
print('→ 1%%p 처럼 작은 차이를 잡으려면 그룹당 1만 명 이상이 필요하다')

### 🖐️ 함께 따라하기 — 더 큰 표본, 더 작은 차이
이번엔 **고객센터 ARS 안내 문구 A/B** 입니다. 안내 문구를 바꾸면 상담사 연결 없이 스스로 해결하는 **자가해결률**이 오를까요? 각 그룹 **2,000건**에 대조군 자가해결 **200건(10%)**, 실험군 **240건(12%)** 이 나왔다고 합시다.

`proportions_ztest` 로 p-value 를, `proportion_effectsize` 로 Cohen's h 를 구해 출력해 보세요. 앞 데모(각 1,000명 · 100건 vs 130건)보다 **그룹당 표본이 2배**인데 **차이는 3%p 에서 2%p 로 더 작다**는 점에 주목하세요 — 표본이 커지면 더 작은 차이도 잡아낼 수 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) n_c, x_c = 2000, 200 (대조군), n_t, x_t = 2000, 240 (실험군) 으로 값을 정한다
# 2) proportions_ztest(count=[x_t, x_c], nobs=[n_t, n_c]) 로 z, p 를 구한다
# 3) proportion_effectsize(x_t/n_t, x_c/n_c) 로 Cohen's h 를 구한다
# 4) p-value 와 h 를 출력하고, p<0.05 인지로 유의성을 판단해 출력한다

### ✅ 바로 확인 퀴즈
**1.** A/B 테스트에서 사용자를 두 그룹에 **무작위로 배정**하는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

무작위 배정을 하면 두 그룹의 다른 조건들이 평균적으로 같아져, 전환율 차이를 **다른 요인이 아니라 '버전 차이'의 효과로 인과적으로 해석**할 수 있기 때문입니다. 이것이 관찰용 상관과 A/B 테스트의 결정적 차이입니다.

</details>

**2.** '결과를 계속 훔쳐보다가 유의해지면 멈추기(peeking)'가 왜 위험한가요?

<details><summary>정답 보기</summary>

우연히 유의해지는 순간을 골라 멈추게 되어 **1종 오류(효과가 없는데 있다고 판정)가 부풀려집니다.** 그래서 실험 **전에** 표본크기를 정하고 그만큼 모을 때까지 결과를 보지 않아야 합니다.

</details>

**3.** `proportion_effectsize` 로 구하는 Cohen's h 는 p-value 와 무엇이 다른가요?

<details><summary>정답 보기</summary>

p-value 는 "차이가 우연인가"(유의성)를 말하고, Cohen's h 는 "**차이가 얼마나 큰가**"(효과크기)를 말합니다. 표본이 크면 아주 작은 차이도 p 가 작아질 수 있으므로, 효과크기를 함께 봐야 **실질적 의미**를 판단할 수 있습니다.

</details>

---
# 4. 통계적 유의 ≠ 실질적 유의 — 표본이 크면 사소한 차이도 유의해진다

A/B 테스트에서 가장 자주 저지르는 오독입니다. **표본이 아주 크면 사업적으로 의미 없는 작은 차이도 `p < 0.05` 가 됩니다.** 그래서 결론은 **p 하나로 내지 않고 효과크기와 함께** 냅니다.

아래는 각 그룹 **10만 명**, 전환율 **12.0% vs 12.4%**(차이 겨우 0.4%p)인 경우입니다.

In [ ]:
# 통계적 유의 != 실질적 유의: 표본이 아주 크면 작은 차이도 '유의'해질 수 있다
# (예: 각 10만 명, 전환율 12.0% vs 12.4% -- 차이는 겨우 0.4%p)
n_big_c, x_big_c = 100000, 12000
n_big_t, x_big_t = 100000, 12400
z_big, p_big = proportions_ztest(count=[x_big_t, x_big_c], nobs=[n_big_t, n_big_c])
h_big = proportion_effectsize(x_big_t / n_big_t, x_big_c / n_big_c)
print('전환율: 대조 %.4f vs 실험 %.4f (차이 %.4f)' %
      (x_big_c / n_big_c, x_big_t / n_big_t, x_big_t / n_big_t - x_big_c / n_big_c))
print('z = %.3f, p-value = %.4f' % (z_big, p_big))
print('효과크기 Cohen h = %.4f (거의 0 = 실질적으로는 무의미)' % h_big)
print('교훈: 0.4%p 처럼 작은 차이도 큰 표본에서는 p<0.05 로 유의해지지만,')
print('     효과크기 h 가 거의 0 이면 실질적 의미는 작다 -- p-value 만 보지 말고 효과크기를 함께 본다')

### 🖐️ 함께 따라하기 — 한 줄 결론 쓰기
2절의 랜딩 페이지 A/B 결과(대조군 10.0% vs 실험군 13.0%, p ≈ 0.036, Cohen's h ≈ 0.09)를 바탕으로, 아래 빈칸을 채운 **의사결정 한 줄**을 `print` 로 출력해 보세요. 정답이 하나로 정해진 문제가 아니라, **유의성(p)·효과크기(h)·사업 판단**을 엮어 말로 정리하는 연습입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 유의성(p-value 로 본 결론), 효과크기(Cohen's h 로 본 크기),
#    사업적 판단(상대 향상 30%)을 한 문장으로 엮어 decision 문자열에 담는다
# 2) print(decision) 으로 출력한다  (정답은 하나가 아니며, 세 요소를 모두 언급하면 좋다)

### ✅ 바로 확인 퀴즈
**1.** 표본이 매우 클 때 p-value 만 보고 의사결정하면 안 되는 이유는?

<details><summary>정답 보기</summary>

표본이 크면 **실질적으로 의미 없는 작은 차이도 통계적으로 유의**해질 수 있기 때문입니다. 따라서 p-value 와 함께 **효과크기(Cohen's h)와 신뢰구간**으로 차이의 실제 크기를 판단해야 합니다.

</details>

**2.** 전환율 10.0% → 13.0% 이고 `p = 0.036`, `Cohen's h = 0.09` 입니다. 한 문장 결론으로 옳은 것은?

<details><summary>정답 보기</summary>

"통계적으로는 유의하지만(p < 0.05) **효과크기는 작다**(h < 0.2)" 입니다. **유의성과 크기는 다른 질문**이라 둘을 함께 말해야 하고, 도입 여부는 여기에 **사업적 판단**(상대 향상 30% 가 비용 대비 가치가 있는가)을 더해 정합니다.

</details>

---
# 🧪 여기서부터 실습 문제 — 실제 A/B 로그로 직접 판단하기

위까지가 **개념과 도구**였습니다. 여기서부터는 **294,478행짜리 실제 이커머스 A/B 로그**(`ab_data.csv`)로 같은 절차를 **처음부터 끝까지 직접** 수행합니다. 위 데모와 달리 원본에는 **오염된 행**이 섞여 있어 **정제부터** 해야 하고, 결과도 데모처럼 깔끔하게 유의하지 않습니다 — 그것이 현실의 A/B 테스트입니다.

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

이 데이터(`ab_data.csv`, 294,478행)는 한 이커머스 A/B 테스트 로그입니다. 각 행은 방문자 한 명이며 `group`(control=기존군 / treatment=실험군), `landing_page`(old_page / new_page), `converted`(0=미전환 / 1=전환)를 담고 있습니다. **원본에는 두 가지 오염**이 있어 정제가 필요합니다 — ① `group` 과 `landing_page` 가 서로 맞지 않는 **불일치 행**, ② 같은 사람이 두 번 기록된 **중복 `user_id`**.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약
#   (미리보기 전용 변수 preview 를 씁니다. 문제 풀이용 df 는 1단계에서 직접 불러오세요.)
preview = pd.read_csv("data/ab_data.csv")
print("행·열 크기:", preview.shape)
print("\n[앞 5행] head()"); display(preview.head())
print("\n[열·자료형·결측] info()"); preview.info()
print("\n[group × landing_page 교차표]"); display(pd.crosstab(preview["group"], preview["landing_page"]))

## 🧪 실습 문제 1 — A/B 전환율 분석: 새 페이지를 도입할까?
**배경**: 제품팀이 새 랜딩 페이지(`new_page`)가 **전환율을 올렸는지** 알고 싶어 합니다. 원본 로그를 정제한 뒤, 두 그룹의 전환율을 비교하고 **비율 검정**과 **신뢰구간**으로 "새 페이지를 전면 도입할지" 를 판단하는 리포트를 완성합니다.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 분석합니다(2단계 정제 결과가 뒤로 이어집니다).

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원본 `(294478, 5)`, 중복 `user_id` 3894개, 그룹↔페이지 불일치 3893개 |
| 2단계 | 정제 후 행수 **290584**, 불일치 0, 중복 `user_id` 0 |
| 3단계 | 전환율 control **0.1204** · treatment **0.1188** |
| 4단계 | 그룹별 전환율 막대그래프 — 완성 그래프처럼 |
| 5단계 | 비율 z-검정 p **0.1899** · Cohen's h **0.0049** |
| 6단계 | 95% 신뢰구간 control [0.1187, 0.1221] · treatment [0.1172, 0.1205] |
| 7단계 | 의사결정 서술(유의하지 않음 → 롤아웃 보류) |

### 1단계 — 데이터 불러오기·구조 파악
`data/ab_data.csv` 를 `df` 로 불러오고, `df.shape`, 중복 `user_id` 개수(`df.duplicated(subset="user_id").sum()`), 그룹↔페이지 불일치 개수를 출력하세요.

- **요구사항**: 원본은 `(294478, 5)`, 중복 `user_id` 는 **3894개**, `group` 과 `landing_page` 가 어긋난 **불일치 행은 3893개**(control 인데 `new_page`, treatment 인데 `old_page`)입니다.
- **불일치 정의**: `treatment` 는 `new_page`, `control` 은 `old_page` 가 정상입니다. 이 짝이 맞지 **않는** 행이 불일치입니다.
- **주의**: 아직 정제하지 않은 **원본 그대로**의 값을 확인하는 단계입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 데이터프레임으로 읽고, 크기·중복 user_id 수·불일치 행 수를 각각 출력한다.

세부구현:
1. read_csv 로 데이터를 df 에 담는다
2. shape 로 행·열 크기를 출력한다
3. duplicated(subset='user_id') 의 합으로 중복 user_id 개수를 센다
4. (group==treatment 이고 landing_page!=new_page) 또는
   (group==control 이고 landing_page!=old_page) 인 행의 개수를 센다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape == (294478, 5)
assert int(df.duplicated(subset="user_id").sum()) == 3894
assert int((((df["group"] == "treatment") & (df["landing_page"] != "new_page")) | ((df["group"] == "control") & (df["landing_page"] != "old_page"))).sum()) == 3893
print("✅ 1단계 통과!")

### 2단계 — 정제 (불일치 제거 → 중복 제거)
같은 `df` 를 다음 **순서**로 정제하세요.

1. **불일치 행 제거**: `group` 과 `landing_page` 가 맞는 행만 남깁니다 — `treatment`↔`new_page`, `control`↔`old_page` 인 행만 유지(불일치 3893행 삭제).
2. **중복 `user_id` 제거**: `drop_duplicates(subset="user_id", keep="first")` 로 같은 사람의 중복 기록을 **첫 행만 남기고** 제거합니다.

- **요구사항**: 정제 후 `df` 의 **행수는 290584**, 불일치 행 0개, 중복 `user_id` 0개여야 합니다.
- **주의**: 불일치를 먼저 제거한 **뒤** 중복을 제거하세요(순서 고정). 삭제 후 `df` 를 그대로 이어 씁니다. (불일치 3893행 제거 → 290585행, 이어서 남은 중복 1행 제거 → 290584행)

<details><summary>힌트</summary>

```text
접근방법:
- 정상 짝인 행만 걸러 남긴 다음, user_id 중복을 첫 행만 남기고 지운다.

세부구현:
1. treatment 이고 new_page 이거나, control 이고 old_page 인 행만 남긴다
2. 그 결과를 다시 df 에 담는다
3. drop_duplicates 에 subset='user_id', keep='first' 를 주어 중복을 지운다
4. 결과를 다시 df 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape[0] == 290584
assert int((((df["group"] == "treatment") & (df["landing_page"] != "new_page")) | ((df["group"] == "control") & (df["landing_page"] != "old_page"))).sum()) == 0
assert int(df.duplicated(subset="user_id").sum()) == 0
print("✅ 2단계 통과!")

### 3단계 — 그룹별 전환율 계산
정제된 `df` 에서 그룹별 **전환율**(= `converted` 의 평균)을 구해 아래 이름의 변수에 담으세요.

- `control_rate` = `control` 그룹의 `converted` 평균
- `treatment_rate` = `treatment` 그룹의 `converted` 평균

- **요구사항(4자리 반올림)**: `control_rate` = **0.1204**, `treatment_rate` = **0.1188**.
- **주의**: `converted` 는 0/1 이므로 **평균이 곧 전환율**입니다. 두 값의 차이가 아주 작다는 점에 주목하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 그룹별로 나눠 converted 의 평균을 구한다.

세부구현:
1. group 이 control 인 행의 converted 평균을 control_rate 에 담는다
2. group 이 treatment 인 행의 converted 평균을 treatment_rate 에 담는다
3. 두 값을 4자리로 반올림해 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(control_rate) - 0.1204) < 0.001
assert abs(float(treatment_rate) - 0.1188) < 0.001
print("✅ 3단계 통과!")

### 4단계 — 그룹별 전환율 막대그래프
두 그룹의 전환율을 **막대그래프**로 그려 한눈에 비교하세요.

- `sns.barplot(data=df, x="group", y="converted", errorbar=None)` 로 그리면 그룹별 평균(=전환율)이 막대로 나옵니다.
- 제목·축 이름을 달고, 그리기 직전에 `plt.figure()` 를 호출하세요.
- 두 막대의 높이가 **거의 같아 보이는 것**이 이 데이터의 핵심입니다(차이가 미미).

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 와 같은 모양으로 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/ab_q1_s4.png" width="520"/>

In [ ]:
# 여기에 코드를 작성하세요

### 5단계 — 비율 검정(z-검정)과 효과크기(Cohen's h)
두 그룹의 전환율 차이가 **통계적으로 유의한지** 비율 z-검정으로 확인하고, 차이의 **크기**를 Cohen's h 로 재세요.

1. 그룹별 **전환 수**(`converted` 합)와 **표본 수**(행 수)를 구합니다.
2. `proportions_ztest(count=[control 전환수, treatment 전환수], nobs=[control 표본수, treatment 표본수])` → 반환값 `(z, p)` 의 `p` 를 `p_value` 에 담습니다.
3. `cohen_h = abs(proportion_effectsize(control_rate, treatment_rate))` — 비율 차이의 효과크기.

- **요구사항(4자리 반올림)**: `p_value` = **0.1899**, `cohen_h` = **0.0049**.
- **해석 기준**: Cohen's h 는 0.2/0.5/0.8 이 small/medium/large 입니다. `p_value` 가 0.05 보다 크고 `cohen_h` 도 0.2 에 한참 못 미치므로 — **차이가 유의하지도, 크지도 않습니다.**
- **주의**: `converted` 는 0/1 이라 **합계가 곧 전환 수**입니다. `proportions_ztest` 는 `(검정통계량, p값)` 순서로 돌려줍니다.

<details><summary>힌트</summary>

```text
접근방법:
- 그룹별 전환 수와 표본 수를 구해 비율 z-검정에 넣고, 효과크기는 두 전환율로 계산한다.

세부구현:
1. control 그룹의 converted 합과 행 수를 구한다
2. treatment 그룹의 converted 합과 행 수를 구한다
3. proportions_ztest 에 count(전환수 목록)와 nobs(표본수 목록)를 넣어 z, p 를 받는다
4. proportion_effectsize(control_rate, treatment_rate) 의 절댓값을 cohen_h 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(p_value) - 0.1899) < 0.01
assert abs(float(cohen_h) - 0.0049) < 0.001
print("✅ 5단계 통과!")

### 6단계 — 그룹별 전환율의 95% 신뢰구간
각 그룹 전환율의 **95% 신뢰구간**을 Wilson 방법으로 구해 아래 변수에 담으세요.

- `ci_control = proportion_confint(control 전환수, control 표본수, method="wilson")`
- `ci_treatment = proportion_confint(treatment 전환수, treatment 표본수, method="wilson")`

- **요구사항(4자리 반올림)**: `ci_control` = **[0.1187, 0.1221]**, `ci_treatment` = **[0.1172, 0.1205]**.
- **해석**: 두 신뢰구간이 **서로 겹칩니다** — 5단계에서 차이가 유의하지 않았던 것과 같은 결론입니다.
- **주의**: `proportion_confint` 는 `(하한, 상한)` 순서로 돌려줍니다. 5단계에서 구한 전환 수·표본 수를 재사용하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 각 그룹의 전환 수와 표본 수로 Wilson 신뢰구간을 구한다.

세부구현:
1. proportion_confint 에 control 전환수·표본수·method='wilson' 을 넣는다
2. proportion_confint 에 treatment 전환수·표본수·method='wilson' 을 넣는다
3. 두 구간을 각각 ci_control, ci_treatment 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(abs(float(x) - e) < 0.001 for x, e in zip(ci_control, [0.1187, 0.1221]))
assert all(abs(float(x) - e) < 0.001 for x, e in zip(ci_treatment, [0.1172, 0.1205]))
print("✅ 6단계 통과!")

### 7단계 — 의사결정 (서술)
위 전환율·검정·신뢰구간을 근거로 **"새 페이지(new_page)를 전면 도입할지"** 에 대한 의사결정을 **3문장 이상** 서술하세요.
- 두 그룹 전환율의 차이, `p_value` 와 유의수준 0.05 의 관계, 효과크기와 신뢰구간이 말하는 바를 근거로 결론을 내리세요.

*(여기에 3문장 이상으로 의사결정을 서술하세요)*

## 🧪 실습 문제 2 — 대용량 표본과 실질적 유의성
**배경**: 이 데이터는 표본이 **29만 건**으로 매우 큽니다. 흔히 "표본이 크면 아주 작은 차이도 통계적으로 유의해진다" 고 합니다. 이번 문제에서는 정제본을 다시 만들어 **전환율 차이의 크기**를 직접 확인하고, **통계적 유의성과 실질적(현업) 유의성의 차이**를 효과크기로 짚은 뒤, "0.01(1%p)의 차이를 검출하려면 표본이 얼마나 필요한가" 를 계산합니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 정제본 재현: 행수 **290584** |
| 2단계 | 전환율 차이 **-0.0016** · 상대변화 **-1.31%** |
| 3단계 | 비율 검정 p **0.1899** · Cohen's h **0.0049** |
| 4단계 | 효과크기 vs 해석 기준 그래프 — 완성 그래프처럼 |
| 5단계 | 통계적 vs 실질적 유의성 서술 |
| 6단계 | MDE 0.01 검출에 필요한 표본 수 **17209** (그룹당) |

### 1단계 — 불러오기·정제 (문제 1 방식 재사용)
문제 2 를 위해 데이터를 **처음부터 다시** 불러와 문제 1 과 **같은 방식**으로 정제합니다.

1. `data/ab_data.csv` 를 `df` 로 불러온다.
2. `treatment`↔`new_page`, `control`↔`old_page` 인 정상 행만 남긴다(불일치 제거).
3. `drop_duplicates(subset="user_id", keep="first")` 로 중복 `user_id` 를 제거한다.

- **요구사항**: 원본을 새로 읽은 직후의 행수를 `raw_rows` 에 담고(**294478**), 정제 후 `df` 의 행수는 **290584** 여야 합니다.
- **주의**: 문제 1 에서 만든 `df` 에 이어 쓰지 말고 **새로 로드**하세요(문제 간 오염 방지). `raw_rows` 는 원본을 실제로 다시 읽었는지 확인하는 값이라, 이어 쓰면 채점을 통과할 수 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 의 2단계 정제를 그대로 다시 수행한다.

세부구현:
1. read_csv 로 원본을 새로 읽는다
2. 정상 짝(treatment-new_page, control-old_page)인 행만 남긴다
3. drop_duplicates(subset='user_id', keep='first') 로 중복을 제거한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert raw_rows == 294478, '원본을 새로 불러왔는지 확인하세요 (문제 1 의 df 를 이어 쓰면 안 됩니다)'
assert df.shape[0] == 290584
print("✅ 1단계 통과!")

### 2단계 — 전환율 차이와 상대 변화율
두 그룹의 전환율을 다시 구하고, **절대 차이**와 **상대 변화율(%)** 을 계산해 아래 변수에 담으세요.

- `control_rate`·`treatment_rate` = 각 그룹의 `converted` 평균
- `abs_diff` = `treatment_rate - control_rate` (실험군 − 대조군, 절대 차이)
- `rel_change` = `abs_diff / control_rate * 100` (대조군 대비 몇 % 변했는지)

- **요구사항(반올림 자리)**: `abs_diff` → 4자리 **-0.0016**, `rel_change` → 2자리 **-1.31**.
- **해석**: 값이 **음수** 라는 것은 실험군 전환율이 오히려 조금 **낮다**는 뜻이고, 상대 변화도 −1.31% 로 미미합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 그룹별 전환율을 구하고, 실험군에서 대조군을 빼 절대 차이를, 그것을 대조군으로 나눠 상대 변화율을 구한다.

세부구현:
1. control_rate, treatment_rate 를 각 그룹 converted 평균으로 구한다
2. abs_diff = treatment_rate - control_rate
3. rel_change = abs_diff / control_rate * 100
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(abs_diff) - (-0.0016)) < 0.001
assert abs(float(rel_change) - (-1.31)) < 0.01
print("✅ 2단계 통과!")

### 3단계 — 비율 검정과 효과크기 다시 확인
"큰 표본이니 이 작은 차이도 유의할까?" 를 직접 확인합니다. 문제 1 과 같은 방식으로 비율 z-검정의 `p_value` 와 효과크기 `cohen_h` 를 구하세요.

1. 그룹별 전환 수·표본 수를 구해 `proportions_ztest` 로 `p_value` 를 얻습니다.
2. `cohen_h = abs(proportion_effectsize(control_rate, treatment_rate))`.

- **요구사항(4자리 반올림)**: `p_value` = **0.1899**, `cohen_h` = **0.0049**.
- **해석**: 표본이 29만 건이나 되는데도 `p_value` 가 0.05 보다 커 **유의하지 않습니다** — "표본이 크면 무조건 유의해진다" 가 항상 참은 아니라는 점, 그리고 효과크기가 결론의 핵심임을 보여줍니다.

<details><summary>힌트</summary>

```text
접근방법:
- 그룹별 전환 수·표본 수로 비율 z-검정을 하고, 두 전환율로 효과크기를 구한다.

세부구현:
1. control/treatment 각 그룹의 converted 합과 행 수를 구한다
2. proportions_ztest 에 count·nobs 를 넣어 p_value 를 받는다
3. proportion_effectsize(control_rate, treatment_rate) 의 절댓값을 cohen_h 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(p_value) - 0.1899) < 0.01
assert abs(float(cohen_h) - 0.0049) < 0.001
print("✅ 3단계 통과!")

### 4단계 — 효과크기와 해석 기준 비교 그래프
관측된 효과크기가 **얼마나 작은지** 를 Cohen's h 의 해석 기준과 함께 막대그래프로 그리세요.

- 막대 하나: 관측 효과크기 `cohen_h`(≈0.0049).
- 가로 기준선 3개: small **0.2**, medium **0.5**, large **0.8** (`ax.axhline(...)`, 점선).
- `ax.set_ylim(0, 0.9)` 로 세로 범위를 고정하면 관측 막대가 기준선에 한참 못 미치는 게 잘 보입니다.
- 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 처럼 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/ab_q2_s4.png" width="520"/>

In [ ]:
# 여기에 코드를 작성하세요

### 5단계 — 통계적 유의성 vs 실질적 유의성 (서술)
위 결과를 근거로 **"통계적으로 유의하다" 와 "실무적으로 의미가 있다" 가 어떻게 다른지** 를 **3문장 이상** 서술하세요.
- 큰 표본에서 p-value 의 성질, 효과크기(Cohen's h)가 왜 필요한지, 이 데이터의 결론(차이도 작고 유의하지도 않음)을 엮으세요.

*(여기에 3문장 이상으로 서술하세요)*

### 6단계 — 표본 크기 산정 (MDE 0.01)
"만약 전환율을 **1%p(0.01)** 올리는 개선을 검출하고 싶다면, 그룹당 표본이 얼마나 필요할까?" 를 계산하세요.

1. 기준 전환율은 `control_rate`(≈0.1204) 로 두고, `es = abs(proportion_effectsize(control_rate, control_rate + 0.01))` 로 검출하려는 효과크기를 구합니다.
2. `n_per_group = NormalIndPower().solve_power(effect_size=es, alpha=0.05, power=0.80, alternative="two-sided")` 로 그룹당 필요한 표본 수를 구하고, `n_ceil = int(np.ceil(n_per_group))` 로 올림합니다.

- **요구사항**: `es` → 4자리 반올림 **0.0302**, `n_ceil` = **17209**(그룹당).
- **해석**: 1%p 라는 작은 개선을 유의수준 0.05·검정력 0.80 으로 잡아내려면 그룹당 약 1.7만 명이 필요합니다 — 이번 실험 표본(그룹당 약 14.5만 명)은 그보다 훨씬 커서, **검출력이 부족해서가 아니라 실제로 차이가 없어서** 유의하지 않았던 것입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 기준 전환율과 목표(기준+0.01)로 효과크기를 구하고, 검정력 분석으로 필요한 표본 수를 계산한다.

세부구현:
1. proportion_effectsize(control_rate, control_rate + 0.01) 의 절댓값을 es 에 담는다
2. NormalIndPower().solve_power 에 effect_size=es, alpha=0.05, power=0.80, alternative='two-sided' 를 넣는다
3. 결과를 올림(np.ceil)해 정수 n_ceil 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(es) - 0.0302) < 0.001
assert n_ceil == 17209
print("✅ 6단계 통과!")

---
## 이 참고 교안 정리

| 개념 | 핵심 | 도구 |
|---|---|---|
| A/B 테스트 | **무작위 배정** 덕분에 차이를 '버전의 효과'로 **인과 해석**할 수 있다 | 실험 설계 |
| 비율 z-검정 | 두 전환율 차이가 우연인가 — **`p` 가 결론**, `z` 는 방향 | `proportions_ztest` |
| 효과크기 | 차이가 **얼마나 큰가**(0.2 작음 / 0.5 중간 / 0.8 큼) — p 와 별개 | `proportion_effectsize` (Cohen's h) |
| 신뢰구간 | 각 그룹 전환율의 **불확실성 폭**. 구간이 겹쳐도 검정은 유의할 수 있다 | `proportion_confint` |
| 표본크기 산정 | 실험 **전에** α·검정력·기준율·MDE 로 정한다. **peeking 금지** | `NormalIndPower().solve_power` |
| 유의 ≠ 실질 | 표본이 크면 사소한 차이도 유의해진다 — **효과크기와 함께** 결론 | p-value + Cohen's h |

**결론은 늘 세 가지를 함께 말합니다** — **유의한가(p) · 얼마나 큰가(효과크기) · 어느 방향인가(실제 전환율)**. 그리고 마지막 한 조각은 통계가 아니라 **사업 판단**입니다: 이 크기의 개선이 도입 비용과 위험을 넘어서는가.

> 실습 문제에서 본 것처럼, 현실의 A/B 로그는 **정제부터** 해야 하고 결과가 **유의하지 않게 나오는 일이 더 흔합니다.** "유의하지 않다" 는 실패가 아니라 **"지금 데이터로는 도입할 근거가 부족하다"** 는 정당한 결론입니다.